# Build and Train SmallLM-8M in JAX

*A complete decoder-only language model built with JAX, Flax NNX, Muon + AdamW, FineWeb-Edu pretraining, SmolTalk instruction tuning, and SafeTensors export.*

This notebook contains the model, optimizer, data preparation, training, SFT, generation, checkpointing, and export implementations. `FULL_RUN = False` is deliberately the default: quick mode exercises the same pipeline with a small token budget; it does not reproduce released metrics.


## 1. Setup

The first cell supports both a repository checkout and a direct Colab upload. A private clone requires normal GitHub authentication; no credential is embedded here.


In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/marcoharuni/small-lm-rs"
if IN_COLAB and not Path("configs/model.json").exists():
    print("Repository files were not found. If the repository is private, authenticate GitHub or upload the checkout/artifact first.")
    CLONE_REPOSITORY = False  # set True after authentication, or when the repository is public
    if CLONE_REPOSITORY:
        subprocess.run(["git", "clone", REPO_URL, "/content/small-lm-rs"], check=True)
        os.chdir("/content/small-lm-rs")

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "jax[cuda12]==0.11.0", "flax==0.12.8", "optax==0.2.8",
        "datasets==5.0.1", "tokenizers==0.23.1", "safetensors==0.8.0",
        "orbax-checkpoint==0.12.2", "numpy==2.5.1", "matplotlib"], check=True)


In [ ]:
import dataclasses, hashlib, importlib.metadata as metadata, json, math, shutil, time
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, asdict
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import orbax.checkpoint as ocp
from datasets import load_dataset
from flax import nnx
from safetensors.numpy import save_file as save_safetensors
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

for package in ("jax", "flax", "optax", "datasets", "tokenizers", "safetensors", "orbax-checkpoint", "numpy", "matplotlib"):
    print(f"{package:18s} {metadata.version(package)}")
print("backend:", jax.default_backend())
print("devices:", jax.devices())
HAS_GPU = any(device.platform == "gpu" for device in jax.devices())
print("GPU ready." if HAS_GPU else "No JAX GPU detected. Inspection/data cells work, but training will be slow; select a Colab GPU runtime.")


## 2. Configuration

These are the released architecture and one-hour pretraining/SFT profiles. Full mode is opt-in.


In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    model_name: str = "nilemini-8m-situ"  # frozen artifact compatibility identifier
    architecture: str = "decoder-only-transformer"
    vocab_size: int = 8192
    context_length: int = 512
    num_layers: int = 8
    hidden_size: int = 256
    intermediate_size: int = 704
    num_query_heads: int = 4
    num_key_value_heads: int = 2
    head_dimension: int = 64
    rms_norm_epsilon: float = 1e-5
    rope_theta: float = 10_000.0
    situ_beta_gate: float = 4.0
    situ_beta_up: float = 25.0
    tie_word_embeddings: bool = True
    use_bias: bool = False
    dropout: float = 0.0
    expected_parameter_count: int = 7_999_744

MODEL = ModelConfig()
assert MODEL.hidden_size == MODEL.num_query_heads * MODEL.head_dimension
assert MODEL.num_query_heads % MODEL.num_key_value_heads == 0

FINEWEB_DATASET, FINEWEB_CONFIG = "HuggingFaceFW/fineweb-edu", "sample-10BT"
FINEWEB_REVISION = "87f09149ef4734204d70ed1d046ddc9ca3f2b8f9"
SMOLTALK_DATASET, SMOLTALK_CONFIG = "HuggingFaceTB/smoltalk", "smol-magpie-ultra"
SMOLTALK_REVISION = "5feaf2fd3ffca7c237fc38d1861bc30365d48ffa"
SPECIAL_TOKENS = ("<|pad|>", "<|bos|>", "<|eos|>", "<|system|>", "<|user|>", "<|assistant|>")
SEED = 42
FULL_RUN = False
ROOT = Path.cwd()
WORK = ROOT / "notebook_work"
ARTIFACT = ROOT / "artifacts" / MODEL.model_name
WORK.mkdir(exist_ok=True)

PRETRAIN_RELEASED = dict(train_tokens=140_017_664, validation_tokens=262_144,
    global_sequences=64, microbatch_sequences=8, validation_sequences=16,
    validate_every=500, checkpoint_every=500, log_every=50,
    muon_lr=0.02, adam_lr=3e-4, weight_decay=0.1, warmup_fraction=0.02, clip_norm=1.0)
SFT_RELEASED = dict(selected_examples=512, train_examples=448, validation_examples=64,
    batch_size=8, microbatch_sequences=1, updates=56, validate_every=28,
    checkpoint_every=28, log_every=10, muon_lr=0.003, adam_lr=5e-5,
    weight_decay=0.01, warmup_fraction=0.02, clip_norm=1.0)
PRETRAIN = PRETRAIN_RELEASED.copy() if FULL_RUN else (PRETRAIN_RELEASED | dict(
    train_tokens=32_768, validation_tokens=4_096, global_sequences=8,
    microbatch_sequences=2, validation_sequences=2, validate_every=1,
    checkpoint_every=1, log_every=1))
PRETRAIN["tokens_per_update"] = PRETRAIN["global_sequences"] * MODEL.context_length
PRETRAIN["updates"] = math.ceil(PRETRAIN["train_tokens"] / PRETRAIN["tokens_per_update"])
assert math.ceil(140_017_664 / (64 * 512)) == 4_273
print("mode:", "released full run" if FULL_RUN else "quick demonstration")
print(PRETRAIN)


## 3. Architecture overview

Tokens pass through a tied FP32 embedding, eight pre-norm residual blocks, and a final RMSNorm. Each block has causal grouped-query attention (4 query heads, 2 KV heads) and the repository's bounded gated FFN. Matmul operands are BF16 with preferred FP32 accumulation.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.3)); ax.axis("off")
labels = ["token IDs", "tied embedding\n8192 × 256", "8 × pre-norm block\nGQA → bounded FFN", "final RMSNorm", "tied LM head\n256 × 8192"]
xs = np.linspace(.08, .92, len(labels))
for x, label in zip(xs, labels):
    ax.text(x, .5, label, ha="center", va="center", bbox=dict(boxstyle="round", fc="#eef4ff", ec="#355c9a"))
for a, b in zip(xs[:-1], xs[1:]): ax.annotate("", (b-.08, .5), (a+.08, .5), arrowprops=dict(arrowstyle="->"))
plt.show()


## 4. Tokenizer and chat format

The released byte-level BPE is used when present. The training functions below are included for a checkout without it: a deterministic 30 MB FineWeb-Edu corpus, byte-level alphabet, and reserved tokens in IDs 0–5.


In [ ]:
def hf_token(): return os.environ.get("HF_TOKEN", "").strip() or None

def fineweb_documents():
    stream = load_dataset(FINEWEB_DATASET, name=FINEWEB_CONFIG, split="train", streaming=True,
                          revision=FINEWEB_REVISION, token=hf_token())
    for row in stream:
        text = str(row.get("text", "")).strip()
        if text:
            doc_id = str(row.get("id") or row.get("url") or hashlib.sha256(text.encode()).hexdigest())
            yield {"id": doc_id, "text": text}

def build_tokenizer_corpus(path, byte_limit=30_000_000):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); written = 0
    with path.open("w", encoding="utf-8") as handle:
        for document in fineweb_documents():
            piece = document["text"] + "\n"; handle.write(piece); written += len(piece.encode("utf-8"))
            if written >= byte_limit: break
    if path.stat().st_size < byte_limit: raise RuntimeError("tokenizer corpus is incomplete")

def train_tokenizer(corpus_path, output_path, min_frequency=2):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=MODEL.vocab_size, min_frequency=min_frequency,
        special_tokens=list(SPECIAL_TOKENS), initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train([str(corpus_path)], trainer=trainer); tok.save(str(output_path)); return tok

def validate_tokenizer(tok):
    assert tok.get_vocab_size() == MODEL.vocab_size
    assert [tok.token_to_id(t) for t in SPECIAL_TOKENS] == list(range(6))
    for s in ("Hello, world!", "Emoji 😀🚀", "Unicode: café 東京 العربية", ""):
        ids = tok.encode(s, add_special_tokens=False).ids
        assert tok.decode(ids, skip_special_tokens=False) == s

tokenizer_path = ARTIFACT / "tokenizer.json"
if not tokenizer_path.exists():
    raise FileNotFoundError("Upload the repository artifact or train the tokenizer with train_tokenizer above.")
tokenizer = Tokenizer.from_file(str(tokenizer_path)); validate_tokenizer(tokenizer)
print("tokenizer sha256:", hashlib.sha256(tokenizer_path.read_bytes()).hexdigest())
sample = "Small models are useful."; ids = tokenizer.encode(sample, add_special_tokens=False).ids
print(ids, "->", tokenizer.decode(ids))


In [ ]:
ROLE_TOKENS = {"system": SPECIAL_TOKENS[3], "user": SPECIAL_TOKENS[4], "assistant": SPECIAL_TOKENS[5]}
def validate_messages(messages):
    if not messages: raise ValueError("messages must not be empty")
    roles = [str(m.get("role", "")) for m in messages]
    dialogue = roles[1:] if roles[0] == "system" else roles
    if roles.count("system") > 1 or "system" in dialogue or not dialogue or dialogue[-1] != "user":
        raise ValueError("optional system first; alternating user/assistant; end with user")
    if any(role != ("user" if i % 2 == 0 else "assistant") for i, role in enumerate(dialogue)):
        raise ValueError("roles must alternate")
    for m in messages:
        content = m.get("content")
        if m.get("role") not in ROLE_TOKENS or not isinstance(content, str) or not content.strip(): raise ValueError("invalid message")
        if any(marker in content for marker in SPECIAL_TOKENS): raise ValueError("reserved marker in content")

def chat_prompt(tok, messages):
    validate_messages(messages); newline = tok.encode("\n", add_special_tokens=False).ids
    ids = [int(tok.token_to_id(SPECIAL_TOKENS[1]))]; eos = int(tok.token_to_id(SPECIAL_TOKENS[2]))
    for m in messages:
        ids += [int(tok.token_to_id(ROLE_TOKENS[m["role"]]))] + newline
        ids += tok.encode(m["content"].strip(), add_special_tokens=False).ids + [eos] + newline
    ids += [int(tok.token_to_id(SPECIAL_TOKENS[5]))] + newline
    if len(ids) >= MODEL.context_length: raise ValueError("prompt leaves no generation room")
    return ids

hello_fixture = chat_prompt(tokenizer, [{"role":"user", "content":"Hello"}])
assert hello_fixture == [1, 4, 204, 45, 474, 84, 2, 204, 5, 204]
print(hello_fixture)


## 5. Core numerical operations and grouped-query attention

`Linear` stores `[in, out]` FP32 kernels. RoPE rotates interleaved even/odd pairs. GQA reshapes queries into two groups per KV head and applies a causal mask before softmax.


In [ ]:
PARAM_DTYPE, COMPUTE_DTYPE = jnp.float32, jnp.bfloat16
def normal_param(rngs, shape, std):
    return nnx.Param((jax.random.normal(rngs.params(), shape, dtype=jnp.float32) * std).astype(PARAM_DTYPE))

def matmul(x, weight):
    a, b = x.astype(COMPUTE_DTYPE), weight.astype(COMPUTE_DTYPE)
    return jax.lax.dot_general(a, b, (((a.ndim - 1,), (0,)), ((), ())), preferred_element_type=jnp.float32)

class Linear(nnx.Module):
    def __init__(self, in_features, out_features, *, rngs, std=.02): self.kernel = normal_param(rngs, (in_features, out_features), std)
    def __call__(self, x): return matmul(x, self.kernel.get_value())

class RMSNorm(nnx.Module):
    def __init__(self, width, epsilon): self.weight, self.epsilon = nnx.Param(jnp.ones(width, jnp.float32)), epsilon
    def __call__(self, x):
        x = x.astype(jnp.float32)
        return x * jax.lax.rsqrt(jnp.mean(jnp.square(x), -1, keepdims=True) + self.epsilon) * self.weight.get_value()

def apply_rope(x, positions, theta):
    d = x.shape[-1]; inv = theta ** (-jnp.arange(0, d, 2, dtype=jnp.float32) / d)
    angles = positions.astype(jnp.float32)[:, None] * inv[None, :]
    c, s = jnp.cos(angles)[None,:,None,:], jnp.sin(angles)[None,:,None,:]
    even, odd = x.astype(jnp.float32)[...,0::2], x.astype(jnp.float32)[...,1::2]
    return jnp.stack((even*c-odd*s, even*s+odd*c), axis=-1).reshape(x.shape)

class GroupedQueryAttention(nnx.Module):
    def __init__(self, config, *, rngs):
        self.config = config; residual_std = .02 / math.sqrt(2 * config.num_layers)
        kv = config.num_key_value_heads * config.head_dimension
        self.q_proj = Linear(config.hidden_size, config.hidden_size, rngs=rngs)
        self.k_proj = Linear(config.hidden_size, kv, rngs=rngs); self.v_proj = Linear(config.hidden_size, kv, rngs=rngs)
        self.o_proj = Linear(config.hidden_size, config.hidden_size, rngs=rngs, std=residual_std)
    def __call__(self, x, positions):
        c = self.config; b,t,_ = x.shape; groups = c.num_query_heads // c.num_key_value_heads
        q = self.q_proj(x).reshape(b,t,c.num_key_value_heads,groups,c.head_dimension)
        k = self.k_proj(x).reshape(b,t,c.num_key_value_heads,c.head_dimension)
        v = self.v_proj(x).reshape(b,t,c.num_key_value_heads,c.head_dimension)
        q = apply_rope(q.reshape(b,t,c.num_query_heads,c.head_dimension), positions, c.rope_theta).reshape(q.shape)
        k = apply_rope(k, positions, c.rope_theta)
        scores = jnp.einsum("btkgh,bskh->bkgts", q.astype(jnp.float32), k.astype(jnp.float32), preferred_element_type=jnp.float32) / math.sqrt(c.head_dimension)
        causal = jnp.arange(t)[:,None] >= jnp.arange(t)[None,:]
        scores = jnp.where(causal[None,None,None,:,:], scores, jnp.finfo(jnp.float32).min)
        attended = jnp.einsum("bkgts,bskh->btkgh", jax.nn.softmax(scores, -1), v.astype(jnp.float32), preferred_element_type=jnp.float32)
        return self.o_proj(attended.reshape(b,t,c.hidden_size))

plt.imshow(np.tril(np.ones((16,16))), cmap="Blues"); plt.title("Causal attention mask"); plt.xlabel("key"); plt.ylabel("query"); plt.show()


## 6. Bounded gated FFN

This is the exact released equation—not SwiGLU or another substitute. The statistic collected during training is the maximum absolute element of `product`.


In [ ]:
class BoundedGatedFFN(nnx.Module):
    def __init__(self, config, *, rngs):
        self.config = config; residual_std = .02 / math.sqrt(2 * config.num_layers)
        self.gate_proj = Linear(config.hidden_size, config.intermediate_size, rngs=rngs)
        self.up_proj = Linear(config.hidden_size, config.intermediate_size, rngs=rngs)
        self.down_proj = Linear(config.intermediate_size, config.hidden_size, rngs=rngs, std=residual_std)
    def __call__(self, x, *, return_product_max=False):
        c = self.config; gate = self.gate_proj(x).astype(jnp.float32); up = self.up_proj(x).astype(jnp.float32)
        gate_branch = c.situ_beta_gate * jnp.tanh(gate / c.situ_beta_gate) * jax.nn.sigmoid(gate)
        up_branch = c.situ_beta_up * jnp.tanh(up / c.situ_beta_up)
        product = gate_branch * up_branch; output = self.down_proj(product)
        return (output, jnp.max(jnp.abs(product))) if return_product_max else output

x = np.linspace(-30,30,500)
plt.plot(x, MODEL.situ_beta_gate*np.tanh(x/MODEL.situ_beta_gate)/(1+np.exp(-x)), label="gate branch")
plt.plot(x, MODEL.situ_beta_up*np.tanh(x/MODEL.situ_beta_up), label="up branch")
plt.axhline(0,color="black",lw=.5); plt.legend(); plt.title("Bounded FFN branches"); plt.show()


## 7. Transformer block and complete model

The output projection reuses the token embedding; there is no separate LM-head parameter.


In [ ]:
class TransformerBlock(nnx.Module):
    def __init__(self, config, *, rngs):
        self.attention_norm = RMSNorm(config.hidden_size, config.rms_norm_epsilon)
        self.attention = GroupedQueryAttention(config, rngs=rngs)
        self.ffn_norm = RMSNorm(config.hidden_size, config.rms_norm_epsilon)
        self.ffn = BoundedGatedFFN(config, rngs=rngs)
    def __call__(self, x, positions, *, collect_stats=False):
        x = x + self.attention(self.attention_norm(x), positions)
        if collect_stats:
            y, maximum = self.ffn(self.ffn_norm(x), return_product_max=True); return x+y, maximum
        return x + self.ffn(self.ffn_norm(x))

class SmallLM(nnx.Module):
    def __init__(self, config=MODEL, *, rngs):
        self.config = config; self.token_embedding = normal_param(rngs, (config.vocab_size, config.hidden_size), .02)
        for i in range(config.num_layers): setattr(self, f"block_{i}", TransformerBlock(config, rngs=rngs))
        self.final_norm = RMSNorm(config.hidden_size, config.rms_norm_epsilon)
    def __call__(self, token_ids, *, collect_stats=False):
        if token_ids.ndim != 2 or token_ids.shape[1] > self.config.context_length: raise ValueError("token IDs must be [batch, sequence] within context")
        positions = jnp.arange(token_ids.shape[1], dtype=jnp.int32); hidden = self.token_embedding.get_value()[token_ids]; maxima=[]
        for i in range(self.config.num_layers):
            if collect_stats: hidden, maximum = getattr(self, f"block_{i}")(hidden, positions, collect_stats=True); maxima.append(maximum)
            else: hidden = getattr(self, f"block_{i}")(hidden, positions)
        logits = matmul(self.final_norm(hidden), self.token_embedding.get_value().T)
        return (logits, jnp.stack(maxima)) if collect_stats else logits

model = SmallLM(rngs=nnx.Rngs(SEED)); graphdef, params = nnx.split(model, nnx.Param)
parameter_count = sum(int(x.size) for x in jax.tree.leaves(params))
assert parameter_count == 7_999_744
kv = MODEL.num_key_value_heads * MODEL.head_dimension
breakdown = {"tied embedding": MODEL.vocab_size*MODEL.hidden_size,
 "attention (8 blocks)": MODEL.num_layers*(2*MODEL.hidden_size**2 + 2*MODEL.hidden_size*kv),
 "FFN (8 blocks)": MODEL.num_layers*3*MODEL.hidden_size*MODEL.intermediate_size,
 "norms": (2*MODEL.num_layers+1)*MODEL.hidden_size}
assert sum(breakdown.values()) == parameter_count
for name,count in breakdown.items(): print(f"{name:24s} {count:>10,}")
print(f"{'total':24s} {parameter_count:>10,}")


## 8. Muon + AdamW

Muon owns only the seven 2-D transformer projections in every block. Embeddings use decayed AdamW; all RMSNorm vectors use AdamW with zero weight decay. Global clipping precedes the partitioned transforms.


In [ ]:
MUON_PATHS = frozenset({("attention","q_proj"),("attention","k_proj"),("attention","v_proj"),("attention","o_proj"),
                                 ("ffn","gate_proj"),("ffn","up_proj"),("ffn","down_proj")})
def path_names(path): return tuple(str(getattr(e,"key",getattr(e,"idx",e))) for e in path)
def parameter_group(path, leaf):
    names=path_names(path); names=names[:-1] if names and names[-1]==".value" else names
    if leaf.ndim==2 and len(names)>=4 and names[0].startswith("block_") and names[-1]=="kernel" and tuple(names[-3:-1]) in MUON_PATHS: return "muon"
    if leaf.ndim==1 and len(names)>=2 and names[-1]=="weight" and "norm" in names[-2]: return "adam_no_decay"
    return "adam_decay"
def labels_for(tree): return jax.tree_util.tree_map_with_path(parameter_group, tree)
def group_counts(tree):
    out={k:0 for k in ("muon","adam_decay","adam_no_decay")}
    for label,leaf in zip(jax.tree.leaves(labels_for(tree)),jax.tree.leaves(tree),strict=True): out[str(label)]+=int(leaf.size)
    return out
def cosine_schedule(peak, updates, fraction):
    warmup=max(1,round(fraction*updates)); return optax.warmup_cosine_decay_schedule(0.,peak,warmup,max(updates,warmup+1),peak*.1)
def build_optimizer(tree, updates, cfg):
    muon_lr=cosine_schedule(cfg["muon_lr"],updates,cfg["warmup_fraction"]); adam_lr=cosine_schedule(cfg["adam_lr"],updates,cfg["warmup_fraction"])
    dims=lambda tree: jax.tree.map(lambda _: optax.contrib.MuonDimensionNumbers(reduction_axis=0,output_axis=1),tree)
    muon=optax.chain(optax.contrib.scale_by_muon(beta=.95,ns_steps=5,nesterov=True,weight_dimension_numbers=dims),
                      optax.add_decayed_weights(cfg["weight_decay"]),optax.scale_by_learning_rate(muon_lr))
    adam=dict(adam_decay=optax.adamw(adam_lr,b1=.9,b2=.95,eps=1e-8,weight_decay=cfg["weight_decay"]),
              adam_no_decay=optax.adamw(adam_lr,b1=.9,b2=.95,eps=1e-8,weight_decay=0.))
    return optax.chain(optax.clip_by_global_norm(cfg["clip_norm"]),optax.multi_transform({"muon":muon}|adam,labels_for(tree)))

counts=group_counts(params); expected={"muon":5_898_240,"adam_decay":2_097_152,"adam_no_decay":4_352}
assert counts==expected and sum(counts.values())==MODEL.expected_parameter_count
print(counts)
steps=np.arange(PRETRAIN["updates"]); plt.plot(steps,[float(cosine_schedule(PRETRAIN["adam_lr"],PRETRAIN["updates"],PRETRAIN["warmup_fraction"])(i)) for i in steps]); plt.title("AdamW learning-rate schedule"); plt.xlabel("update"); plt.show()


## 9. FineWeb-Edu preparation

Documents are assigned by the first eight SHA-256 bytes of the document ID: buckets 0–99 of 10,000 are validation. Each document receives EOS, then streams are packed contiguously as little-endian `uint16`. An extra token supports next-token shifting. Quick mode avoids materializing 140M tokens.


In [ ]:
def document_split(document_id):
    value=int.from_bytes(hashlib.sha256(document_id.encode()).digest()[:8],"big")
    return "validation" if value % 10_000 < 100 else "train"

def prepare_pretraining_data(train_tokens, validation_tokens, output_dir):
    output_dir.mkdir(parents=True,exist_ok=True); paths={s:output_dir/f"{s}.bin" for s in ("train","validation")}
    targets={"train":train_tokens+1,"validation":validation_tokens+1}; counts={"train":0,"validation":0}; eos=int(tokenizer.token_to_id(SPECIAL_TOKENS[2]))
    with paths["train"].open("wb") as train_file, paths["validation"].open("wb") as val_file:
        files={"train":train_file,"validation":val_file}
        for document in fineweb_documents():
            split=document_split(document["id"]); remaining=targets[split]-counts[split]
            if remaining<=0:
                if counts==targets: break
                continue
            ids=tokenizer.encode(document["text"],add_special_tokens=False).ids+[eos]
            array=np.asarray(ids[:remaining],dtype="<u2"); array.tofile(files[split]); counts[split]+=array.size
            if counts==targets: break
    if counts!=targets: raise RuntimeError(f"incomplete data: {counts}")
    manifest={"dataset":FINEWEB_DATASET,"config":FINEWEB_CONFIG,"revision":FINEWEB_REVISION,
              "tokenizer_sha256":hashlib.sha256(tokenizer_path.read_bytes()).hexdigest(),"train_tokens":train_tokens,"validation_tokens":validation_tokens,"dtype":"uint16"}
    (output_dir/"manifest.json").write_text(json.dumps(manifest,indent=2)+"\n"); return paths

class TokenBatcher:
    def __init__(self,path,target_tokens): self.tokens=np.memmap(path,mode="r",dtype="<u2"); self.target_tokens=target_tokens
    def batch(self,update,sequences):
        length=MODEL.context_length; start_target=update*sequences*length; valid=min(max(0,self.target_tokens-start_target),sequences*length)
        inputs=np.zeros((sequences,length),np.int32); targets=np.zeros_like(inputs); mask=np.zeros_like(inputs,np.float32)
        for row in range(math.ceil(valid/length) if valid else 0):
            start=start_target+row*length; take=min(length,self.target_tokens-start); segment=np.asarray(self.tokens[start:start+take+1],np.int32)
            inputs[row,:take],targets[row,:take],mask[row,:take]=segment[:-1],segment[1:],1.
        return inputs,targets,mask
    def batches(self,sequences):
        for update in range(math.ceil(self.target_tokens/(sequences*MODEL.context_length))): yield self.batch(update,sequences)

# Network/data write is explicit, not automatic on Run All.
PREPARE_DATA = False
DATA_DIR=WORK/("pretrain_full" if FULL_RUN else "pretrain_quick")
if PREPARE_DATA: data_paths=prepare_pretraining_data(PRETRAIN["train_tokens"],PRETRAIN["validation_tokens"],DATA_DIR)
else: print("Set PREPARE_DATA=True to materialize the selected token budget.")


## 10. Loss, gradient accumulation, evaluation, and checkpoints


In [ ]:
def token_loss_sum(logits,targets,mask):
    losses=optax.softmax_cross_entropy_with_integer_labels(logits.astype(jnp.float32),targets)
    return jnp.sum(losses*mask),jnp.sum(mask),jnp.sum((jnp.argmax(logits,-1)==targets)*mask)
def compile_loss_and_grad(graphdef):
    @jax.jit
    def step(tree,inputs,targets,mask):
        def objective(candidate):
            logits,maxima=nnx.merge(graphdef,candidate)(inputs,collect_stats=True); loss,valid,correct=token_loss_sum(logits,targets,mask)
            return loss,(valid,correct,jnp.max(maxima))
        return jax.value_and_grad(objective,has_aux=True)(tree)
    return step
def compile_apply(optimizer):
    @jax.jit
    def apply(tree,state,gradients):
        updates,next_state=optimizer.update(gradients,state,tree); return optax.apply_updates(tree,updates),next_state
    return apply
def optimizer_update(tree,state,batch,loss_and_grad,apply_gradients,microbatch_size):
    inputs,targets,mask=batch; gradient_sum=jax.tree.map(jnp.zeros_like,tree)
    loss=valid=correct=maximum=jnp.asarray(0.,jnp.float32)
    for start in range(0,inputs.shape[0],microbatch_size):
        micro_mask=mask[start:start+microbatch_size]
        if not np.any(micro_mask): continue
        (micro_loss,(micro_valid,micro_correct,micro_max)),grads=loss_and_grad(tree,jnp.asarray(inputs[start:start+microbatch_size]),jnp.asarray(targets[start:start+microbatch_size]),jnp.asarray(micro_mask))
        gradient_sum=jax.tree.map(lambda a,b:a+b,gradient_sum,grads); loss+=micro_loss; valid+=micro_valid; correct+=micro_correct; maximum=jnp.maximum(maximum,micro_max)
    if float(valid)==0: raise RuntimeError("no valid targets")
    gradients=jax.tree.map(lambda g:g/valid,gradient_sum); grad_norm=optax.global_norm(gradients); tree,state=apply_gradients(tree,state,gradients)
    metrics={"loss":float(loss)/float(valid),"accuracy":float(correct)/float(valid),"tokens":int(valid),"gradient_norm":float(grad_norm),"bounded_ffn_product_max":float(maximum)}
    if not all(math.isfinite(metrics[k]) for k in ("loss","gradient_norm","bounded_ffn_product_max")): raise FloatingPointError("non-finite training metric")
    return tree,state,metrics
def compile_evaluation_step(graphdef):
    @jax.jit
    def step(tree,inputs,targets,mask): return token_loss_sum(nnx.merge(graphdef,tree)(inputs),targets,mask)
    return step
def evaluate(tree,batcher,step,sequences):
    loss=tokens=correct=0.
    for x,y,m in batcher.batches(sequences):
        a,b,c=step(tree,jnp.asarray(x),jnp.asarray(y),jnp.asarray(m)); loss+=float(a); tokens+=float(b); correct+=float(c)
    mean=loss/tokens; return {"loss":mean,"perplexity":math.exp(min(mean,20.)),"accuracy":correct/tokens}

def save_checkpoint(directory,payload,step,keep=2):
    directory.mkdir(parents=True,exist_ok=True); path=directory/f"step-{step:08d}"
    with ocp.StandardCheckpointer() as ckpt: ckpt.save(path,payload,force=True); ckpt.wait_until_finished()
    completed=sorted((p for p in directory.glob("step-*")),reverse=True)
    for stale in completed[keep:]: shutil.rmtree(stale)
    return path
def latest_checkpoint(directory):
    paths=sorted(directory.glob("step-*")); return paths[-1] if paths else None
def restore_checkpoint(path,target):
    with ocp.StandardCheckpointer() as ckpt: return ckpt.restore(path,target)


## 11. Pretraining loop

This is resumable and writes actual JSONL records. It runs only after data preparation and explicit `RUN_PRETRAINING=True`.

> **Reference result from the released run:** base validation loss **3.8659**, perplexity **47.74**. These are historical values, not notebook outputs.


In [ ]:
def run_pretraining(tree,graphdef,train_batcher,validation_batcher,cfg,run_dir):
    run_dir.mkdir(parents=True,exist_ok=True); history_path=run_dir/"metrics.jsonl"; checkpoint_dir=run_dir/"checkpoints"
    optimizer=build_optimizer(tree,cfg["updates"],cfg); state=optimizer.init(tree); loss_grad=compile_loss_and_grad(graphdef); apply=compile_apply(optimizer); eval_step=compile_evaluation_step(graphdef)
    start=tokens_processed=0; latest=latest_checkpoint(checkpoint_dir)
    if latest:
        restored=restore_checkpoint(latest,{"params":tree,"optimizer":state,"step":0,"tokens_processed":0})
        tree,state,start,tokens_processed=restored["params"],restored["optimizer"],int(restored["step"]),int(restored["tokens_processed"])
    began=log_time=time.perf_counter(); log_tokens=tokens_processed
    for update in range(start,cfg["updates"]):
        tree,state,metrics=optimizer_update(tree,state,train_batcher.batch(update,cfg["global_sequences"]),loss_grad,apply,cfg["microbatch_sequences"])
        completed=update+1; tokens_processed+=metrics["tokens"]; now=time.perf_counter(); metrics["tokens_per_second"]=(tokens_processed-log_tokens)/max(now-log_time,1e-9)
        with history_path.open("a") as f: f.write(json.dumps({"kind":"train","update":completed,"tokens_processed":tokens_processed,**metrics})+"\n")
        if completed%cfg["log_every"]==0 or completed==1:
            print(f"{completed:,}/{cfg['updates']:,} loss={metrics['loss']:.4f} grad={metrics['gradient_norm']:.3f} tok/s={metrics['tokens_per_second']:,.0f}"); log_time,log_tokens=now,tokens_processed
        if completed%cfg["validate_every"]==0 or completed==cfg["updates"]:
            validation=evaluate(tree,validation_batcher,eval_step,cfg["validation_sequences"])
            with history_path.open("a") as f: f.write(json.dumps({"kind":"validation","update":completed,"tokens_processed":tokens_processed,**validation})+"\n")
            print("validation",validation)
        if completed%cfg["checkpoint_every"]==0 or completed==cfg["updates"]:
            save_checkpoint(checkpoint_dir,{"params":tree,"optimizer":state,"step":completed,"tokens_processed":tokens_processed},completed)
    assert tokens_processed==cfg["train_tokens"]
    return tree,history_path

RUN_PRETRAINING=False
if RUN_PRETRAINING:
    if not HAS_GPU: raise RuntimeError("Select a GPU runtime before training")
    data_paths={s:DATA_DIR/f"{s}.bin" for s in ("train","validation")}
    params,pretrain_history=run_pretraining(params,graphdef,TokenBatcher(data_paths["train"],PRETRAIN["train_tokens"]),TokenBatcher(data_paths["validation"],PRETRAIN["validation_tokens"]),PRETRAIN,WORK/"run_pretrain")


## 12. Training curves

Plots consume only records written by the loop; the notebook contains no fabricated outputs.


In [ ]:
def plot_history(path):
    records=[json.loads(line) for line in Path(path).read_text().splitlines()]; train=[r for r in records if r["kind"]=="train"]; val=[r for r in records if r["kind"]=="validation"]
    fig,axes=plt.subplots(2,3,figsize=(14,7)); series=[("loss","train loss"),("gradient_norm","gradient norm"),("bounded_ffn_product_max","bounded FFN product max"),("tokens_per_second","tokens/sec")]
    for ax,(key,title) in zip(axes.flat,series): ax.plot([r["update"] for r in train],[r[key] for r in train]); ax.set_title(title)
    axes.flat[4].plot([r["update"] for r in val],[r["loss"] for r in val],marker="o"); axes.flat[4].set_title("validation loss")
    axes.flat[5].plot([r["update"] for r in val],[r["perplexity"] for r in val],marker="o"); axes.flat[5].set_title("validation perplexity")
    for ax in axes.flat: ax.set_xlabel("update"); ax.grid(alpha=.2)
    plt.tight_layout(); plt.show()
# plot_history(pretrain_history)


## 13. SmolTalk supervised fine-tuning

The pinned stream rejects tools/images, malformed turns, reserved markers, duplicates, empty content, and examples longer than 513 tokens before shifting. It selects the first 512 valid unique conversations, then uses 448 train and 64 validation examples. Loss covers assistant text and assistant EOS only.

> **Reference result from the released run:** SFT validation loss **2.6459**. This is historical, not an output from quick mode.


In [ ]:
def append_tokens(ids,mask,values,include_loss): ids.extend(map(int,values)); mask.extend([int(include_loss)]*len(values))
def format_conversation(tok,messages):
    roles=[str(m.get("role","")) for m in messages]; dialogue=roles[1:] if roles and roles[0]=="system" else roles
    valid=len(dialogue)>=2 and dialogue[-1]=="assistant" and all(r==("user" if i%2==0 else "assistant") for i,r in enumerate(dialogue)) and roles.count("system")<=1
    if not valid:return None
    bos,eos,pad=[int(tok.token_to_id(SPECIAL_TOKENS[i])) for i in (1,2,0)]; newline=tok.encode("\n",add_special_tokens=False).ids; ids=[bos]; mask=[0]
    for message in messages:
        role=str(message.get("role","")); content=message.get("content")
        if role not in ROLE_TOKENS or not isinstance(content,str) or not content.strip() or any(t in content for t in SPECIAL_TOKENS): return None
        append_tokens(ids,mask,[tok.token_to_id(ROLE_TOKENS[role])],False); append_tokens(ids,mask,newline,False)
        append_tokens(ids,mask,tok.encode(content.strip(),add_special_tokens=False).ids,role=="assistant"); append_tokens(ids,mask,[eos],role=="assistant"); append_tokens(ids,mask,newline,False)
    if len(ids)>MODEL.context_length+1 or sum(mask)==0:return None
    padded=np.full(MODEL.context_length+1,pad,np.uint16); loss=np.zeros(MODEL.context_length+1,np.uint8); padded[:len(ids)]=ids; loss[:len(mask)]=mask
    return padded[:-1],padded[1:],loss[1:]

def prepare_sft_data(output_dir,profile=SFT_RELEASED):
    output_dir.mkdir(parents=True,exist_ok=True)
    stream=load_dataset(SMOLTALK_DATASET,name=SMOLTALK_CONFIG,split="train",streaming=True,revision=SMOLTALK_REVISION,token=hf_token())
    examples=[]; seen=set()
    for row in stream:
        messages=row.get("messages")
        if not isinstance(messages,list) or row.get("tools") or row.get("images") or row.get("image"):continue
        digest=hashlib.sha256(json.dumps(messages,sort_keys=True,ensure_ascii=False,separators=(",",":")).encode()).digest()
        if digest in seen:continue
        formatted=format_conversation(tokenizer,messages)
        if formatted is None:continue
        seen.add(digest); examples.append(formatted)
        if len(examples)==profile["selected_examples"]:break
    if len(examples)!=profile["selected_examples"]:raise RuntimeError("insufficient valid SFT examples")
    for name,rows in (("train",examples[:profile["train_examples"]]),("validation",examples[profile["train_examples"]:])):
        for field,index in (("inputs",0),("targets",1),("mask",2)):np.save(output_dir/f"{name}_{field}.npy",np.stack([row[index] for row in rows]))

class SFTBatcher:
    def __init__(self,directory,split,seed=SEED):
        self.inputs=np.load(directory/f"{split}_inputs.npy");self.targets=np.load(directory/f"{split}_targets.npy");self.mask=np.load(directory/f"{split}_mask.npy");self.seed=seed
    def epoch(self,batch_size,start_update=0):
        order=np.random.default_rng(self.seed).permutation(len(self.inputs))
        for update,start in enumerate(range(0,len(order),batch_size)):
            if update<start_update:continue
            idx=order[start:start+batch_size]; x=np.zeros((batch_size,MODEL.context_length),np.int32);y=np.zeros_like(x);m=np.zeros_like(x,np.float32)
            x[:len(idx)],y[:len(idx)],m[:len(idx)]=self.inputs[idx],self.targets[idx],self.mask[idx];yield x,y,m
    def batches(self,batch_size):
        for start in range(0,len(self.inputs),batch_size):yield self.inputs[start:start+batch_size].astype(np.int32),self.targets[start:start+batch_size].astype(np.int32),self.mask[start:start+batch_size].astype(np.float32)


In [ ]:
def run_sft(tree,graphdef,directory,cfg=SFT_RELEASED):
    cfg=cfg.copy(); optimizer=build_optimizer(tree,cfg["updates"],cfg); state=optimizer.init(tree); loss_grad=compile_loss_and_grad(graphdef); apply=compile_apply(optimizer); eval_step=compile_evaluation_step(graphdef)
    train,validation=SFTBatcher(directory,"train"),SFTBatcher(directory,"validation"); history=WORK/"sft_metrics.jsonl"
    for completed,batch in enumerate(train.epoch(cfg["batch_size"]),start=1):
        tree,state,metrics=optimizer_update(tree,state,batch,loss_grad,apply,cfg["microbatch_sequences"])
        with history.open("a") as f:f.write(json.dumps({"kind":"train","update":completed,**metrics})+"\n")
        if completed%cfg["log_every"]==0 or completed==1:print(completed,metrics)
        if completed%cfg["validate_every"]==0 or completed==cfg["updates"]:
            result=evaluate(tree,validation,eval_step,cfg["batch_size"])
            with history.open("a") as f:f.write(json.dumps({"kind":"validation","update":completed,**result})+"\n")
        if completed%cfg["checkpoint_every"]==0 or completed==cfg["updates"]:save_checkpoint(WORK/"sft_checkpoints",{"params":tree,"optimizer":state,"step":completed},completed)
    return tree,history

PREPARE_SFT=False; RUN_SFT=False; SFT_DIR=WORK/"sft_data"
if PREPARE_SFT:prepare_sft_data(SFT_DIR)
if RUN_SFT:
    if not HAS_GPU:raise RuntimeError("Select a GPU runtime")
    params,sft_history=run_sft(params,graphdef,SFT_DIR)


## 14. Generate and chat

Sampling is uncached and intended as a clear JAX reference. Greedy mode is `temperature=0`; otherwise top-k and top-p filters are applied before categorical sampling.


In [ ]:
def sample_token(logits,key,temperature=0.,top_k=None,top_p=None):
    if temperature<=0:return int(jnp.argmax(logits))
    logits=logits.astype(jnp.float32)/temperature
    if top_k is not None:
        threshold=jnp.sort(logits)[-min(top_k,logits.size)];logits=jnp.where(logits>=threshold,logits,-jnp.inf)
    if top_p is not None and top_p<1:
        order=jnp.argsort(logits)[::-1]; sorted_logits=logits[order]; probs=jax.nn.softmax(sorted_logits); remove=jnp.cumsum(probs)>top_p;remove=remove.at[0].set(False)
        logits=logits.at[order].set(jnp.where(remove,-jnp.inf,sorted_logits))
    return int(jax.random.categorical(key,logits))
def generate(tree,messages,max_new_tokens=64,temperature=0.,top_k=50,top_p=.95,seed=0):
    ids=chat_prompt(tokenizer,messages);eos=int(tokenizer.token_to_id(SPECIAL_TOKENS[2]));local_model=nnx.merge(graphdef,tree);generated=[];key=jax.random.key(seed)
    for _ in range(max_new_tokens):
        if len(ids)>=MODEL.context_length:break
        key,subkey=jax.random.split(key);next_id=sample_token(local_model(jnp.asarray(ids,jnp.int32)[None,:])[0,-1],subkey,temperature,top_k,top_p)
        if next_id==eos:break
        ids.append(next_id);generated.append(next_id)
    return tokenizer.decode(generated,skip_special_tokens=True),generated

# Interactive use after training or restore:
# prompt=input("You: ")
# print("SmallLM:",generate(params,[{"role":"user","content":prompt}],temperature=.8)[0])


## 15. Evaluation and checkpoint restore

Evaluate validation loss/perplexity with `evaluate`. Sample generations are qualitative only: this is an 8M-parameter model, not a strong general assistant, and no benchmark claim is implied.


In [ ]:
def save_parameters(path,tree):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with ocp.StandardCheckpointer() as ckpt:ckpt.save(path,tree,force=True);ckpt.wait_until_finished()
    return path
def restore_parameters(path,target):
    with ocp.StandardCheckpointer() as ckpt:return ckpt.restore(Path(path),target)

# save_parameters(WORK/"final-params",params)
# params=restore_parameters(WORK/"final-params",params)
# print(evaluate(params, validation_batcher, compile_evaluation_step(graphdef), PRETRAIN["validation_sequences"]))


## 16. SafeTensors export and JAX parity fixture

Linear kernels transpose from training `[in, out]` to Rust `[out, in]`. The tied embedding is exported once. The fixture stores the exact chat prompt and FP32 JAX logits; Rust parity remains a Rust-engine test.


In [ ]:
def export_tensors(tree):
    pure=nnx.to_pure_dict(tree);tensors={"token_embedding.weight":np.asarray(pure["token_embedding"],np.float32)}
    def linear(name,node):tensors[name]=np.asarray(node["kernel"],np.float32).T.copy()
    for layer in range(MODEL.num_layers):
        block=pure[f"block_{layer}"];prefix=f"layers.{layer}";tensors[f"{prefix}.attention_norm.weight"]=np.asarray(block["attention_norm"]["weight"],np.float32)
        for short in ("q_proj","k_proj","v_proj","o_proj"):linear(f"{prefix}.{short}.weight",block["attention"][short])
        tensors[f"{prefix}.ffn_norm.weight"]=np.asarray(block["ffn_norm"]["weight"],np.float32)
        for short in ("gate_proj","up_proj","down_proj"):linear(f"{prefix}.{short}.weight",block["ffn"][short])
    tensors["final_norm.weight"]=np.asarray(pure["final_norm"]["weight"],np.float32)
    assert sum(x.size for x in tensors.values())==MODEL.expected_parameter_count;return tensors
def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):digest.update(chunk)
    return digest.hexdigest()
def export_model(tree,output):
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    save_safetensors(export_tensors(tree),str(output/"model.safetensors"),metadata={"format":"nilemini-v1","dtype":"float32","tied_embeddings":"true"})
    shutil.copy2(tokenizer_path,output/"tokenizer.json")
    config=asdict(MODEL)|{"normalization":"RMSNorm","position_encoding":"RoPE","activation":"SiTU-GLU","weight_layout":"out_features,in_features"}
    (output/"config.json").write_text(json.dumps(config,indent=2)+"\n")
    (output/"generation_config.json").write_text(json.dumps({"context_length":MODEL.context_length,"pad_token_id":0,"eos_token_id":2},indent=2)+"\n")
    reference=np.asarray(chat_prompt(tokenizer,[{"role":"user","content":"Hello"}]),np.int32)[None,:]
    logits=np.asarray(nnx.merge(graphdef,tree)(jnp.asarray(reference)),np.float32)
    (output/"reference_inputs.json").write_text(json.dumps({"token_ids":reference.tolist()},indent=2)+"\n")
    save_safetensors({"logits":logits},str(output/"reference_outputs.safetensors"))
    files=sorted(p for p in output.iterdir() if p.is_file() and p.name not in {"manifest.json","SHA256SUMS"});checks={p.name:file_sha256(p) for p in files}
    manifest={"model_name":MODEL.model_name,"parameter_count":MODEL.expected_parameter_count,"tokenizer_sha256":file_sha256(output/"tokenizer.json"),"files":checks}
    (output/"manifest.json").write_text(json.dumps(manifest,indent=2,sort_keys=True)+"\n");(output/"SHA256SUMS").write_text("\n".join(f"{v}  {k}" for k,v in sorted(checks.items()))+"\n")
    return output

# exported=export_model(params,WORK/"export" )
# !cargo run --release -p nilemini-engine --example parity -- artifacts/nilemini-8m-situ


## 17. Final usage

1. Set `PREPARE_DATA=True` and run the data cell.
2. Set `RUN_PRETRAINING=True`; optionally prepare and run SFT.
3. Save or restore with the Orbax helpers.
4. Call `generate(...)` for local chat.
5. Call `export_model(params, WORK / "export")` for the Rust-compatible package.

Quick mode follows the same code path with a small budget. Set `FULL_RUN=True` before running configuration and later cells to select the exact released 140,017,664-token, 4,273-update profile.
